> ## SOLUTION / ANSWER KEY &mdash; Lab 2.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-overfitting-early-stopping.ipynb`](../lab-08-overfitting-early-stopping.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 2.8 &mdash; Overfitting & Early Stopping

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Introduction to Deep Learning**

### What you'll do
- Measure the gap between training and validation accuracy
- See a high-capacity network memorise noisy training data
- Use early stopping to halt before it overfits

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 2 slides &mdash; Fighting overfitting](../../presentation/day1-module2-introduction-to-deep-learning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 2 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # quiet TensorFlow logs (used in the advanced labs)
WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-02-08")
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
A network with too much capacity **memorises** the training data -- including its noise -- and
fails on new data. The **train/validation gap** exposes it. **Early stopping** halts training
once validation stops improving.

> Needs `scikit-learn`.

In [ ]:
# DEMO -- a small, noisy dataset and a held-out validation set
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=400, n_features=20, n_informative=5, flip_y=0.2, random_state=0)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.4, random_state=0)
print("train:", len(X_tr), "validation:", len(X_val))

In [ ]:
# DEMO -- visualise the 20-feature input as a 2D PCA projection
try:
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA
    XY = PCA(n_components=2, random_state=0).fit_transform(X)   # 20-D -> 2-D for plotting only
    plt.scatter(XY[y == 0, 0], XY[y == 0, 1], s=20, label="class 0")
    plt.scatter(XY[y == 1, 0], XY[y == 1, 1], s=20, label="class 1")
    plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
    plt.title("make_classification input (20-D, shown in 2-D via PCA)")
    plt.legend(); plt.tight_layout()
    plt.savefig(WORK + "/classification_input.png", dpi=90); plt.show()
    print("saved:", WORK + "/classification_input.png")
except Exception as e:
    print("Plot needs matplotlib (pip install matplotlib).", type(e).__name__)

## Your Turn
Compute the validation accuracy of an overfit network, and turn **early stopping** on.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# a big network, trained hard -> memorises the training set
big = MLPClassifier(hidden_layer_sizes=(200, 200), max_iter=2000, random_state=0).fit(X_tr, y_tr)
def train_acc(): return accuracy_score(y_tr, big.predict(X_tr))
def val_acc():
    return accuracy_score(y_val, big.predict(X_val))
def gap(): return train_acc() - val_acc()

def make_early_stopped():
    clf = MLPClassifier(hidden_layer_sizes=(200, 200), max_iter=500,
                        early_stopping=True, random_state=0)
    return clf.fit(X_tr, y_tr)

try:
    es = make_early_stopped()
    print(f'train={train_acc():.2f} val={val_acc():.2f} gap={gap():.2f} | early-stopped at {es.n_iter_} iters')
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

from sklearn.neural_network import MLPClassifier
expect_true("training accuracy is very high (memorised)", lambda: train_acc() >= 0.95)
expect_true("validation accuracy is clearly lower (the gap)", lambda: gap() > 0.1)
expect_true("early stopping halts before max_iter (500)", lambda: make_early_stopped().n_iter_ < 500)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
More capacity is not always better. The validation gap is your early-warning system; early stopping is one of the cheapest cures.

[&#8592; All Module 2 labs](./index.html) &nbsp;&middot;&nbsp; [Module 2 slides](../../presentation/day1-module2-introduction-to-deep-learning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>